# Лабораторна робота №4: Статистичний аналіз та перевірка гіпотез

### Мета роботи

Навчитися формулювати статистичні гіпотези, обирати та застосовувати відповідний статистичний тест, інтерпретувати результати та приймати обґрунтовані рішення на основі даних. Окремий акцент — проведення A/B-тестування як типової задачі аналітика даних.

### Варіант завдання

Варіант визначається за номером студента у списку групи. Кожен варіант задає **статистичний тест**, який потрібно застосувати у Розділі 2.

| Номер у списку | Тест | Дослідницьке питання (Penguins) |
|:-:|---|---|
| 1, 4, 7, 10, 13, 16, 19, 22, 25, 28 | Одновибірковий t-тест | Чи відрізняється середня маса тіла пінгвінів Аделі від 3700 г? |
| 2, 5, 8, 11, 14, 17, 20, 23, 26, 29 | Двовибірковий t-тест (Велча) | Чи відрізняється середня довжина ласт між пінгвінами Аделі та Генту? |
| **3, 6, 9, 12, 15, 18, 21, 24, 27, 30** | **Хі-квадрат тест** | **Чи є зв'язок між видом пінгвіна та островом проживання?** |

### Набір даних: Palmer Penguins

Набір даних містить 344 записи про пінгвінів з архіпелагу Палмер, Антарктида. Зібраний дослідницькою групою станції Палмер (Gorman, Williams & Fraser, 2014). Класичний навчальний датасет для курсів статистики та машинного навчання.

| Колонка | Тип | Опис |
|---------|-----|------|
| `species` | категоріальний | Вид пінгвіна: Adelie, Chinstrap, Gentoo |
| `island` | категоріальний | Острів: Biscoe, Dream, Torgersen |
| `bill_length_mm` | числовий | Довжина дзьоба (мм) |
| `bill_depth_mm` | числовий | глибина дзьоба (мм) |
| `flipper_length_mm` | числовий | Довжина ласти (мм) |
| `body_mass_g` | числовий | Маса тіла (г) |
| `sex` | категоріальний | Стать: Male, Female |

### Підготовка середовища та завантаження даних

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

# Завантаження Palmer Penguins
df = sns.load_dataset('penguins')

print(f'Дані завантажено: {df.shape}')
df.head()

---

## Розвідувальний аналіз та формулювання гіпотез

Перед проведенням будь-якого статистичного тесту необхідно зрозуміти дані та чітко сформулювати гіпотези.

**Вимоги:**

1. Визначте змінні, що відповідають вашому варіанту. Виведіть описову статистику саме для цих змінних.
2. Побудуйте візуалізацію, що демонструє відмінність або зв'язок, які ви будете тестувати. Тип графіка оберіть самостійно — він має відповідати типу змінних у вашому варіанті.
3. У текстовій клітинці чітко сформулюйте:
   - **H₀** (нульову гіпотезу) — що саме ви перевіряєте
   - **H₁** (альтернативну гіпотезу) — що ви очікуєте знайти
   - рівень значущості α
   - обґрунтування: чому саме цей тест підходить для вашого питання

In [ ]:
# 1.1. Описова статистика для змінних вашого варіанту
# Варіант 3: змінні species та island (обидві категоріальні)

print('=== Розподіл за видом пінгвіна (species) ===')
print(df['species'].value_counts())
print(f'\nЧастки:\n{df["species"].value_counts(normalize=True).round(3)}')

print('\n=== Розподіл за островом (island) ===')
print(df['island'].value_counts())
print(f'\nЧастки:\n{df["island"].value_counts(normalize=True).round(3)}')

print('\n=== Таблиця спряженості (вид × острів) ===')
contingency_table = pd.crosstab(df['species'], df['island'])
print(contingency_table)

print('\n=== Таблиця спряженості (відсотки по рядках) ===')
print(pd.crosstab(df['species'], df['island'], normalize='index').round(3) * 100)

In [ ]:
# 1.2. Візуалізація: зв'язок між видом пінгвіна та островом проживання

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Графік 1: Grouped bar chart
contingency_table.plot(kind='bar', ax=axes[0], colormap='Set2', edgecolor='white', linewidth=0.8)
axes[0].set_title('Кількість пінгвінів за видом та островом')
axes[0].set_xlabel('Вид пінгвіна')
axes[0].set_ylabel('Кількість особин')
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(title='Острів')

# Графік 2: Stacked bar chart (нормалізований — частки)
ct_norm = pd.crosstab(df['species'], df['island'], normalize='index') * 100
ct_norm.plot(kind='bar', stacked=True, ax=axes[1], colormap='Set2', edgecolor='white', linewidth=0.5)
axes[1].set_title('Частка островів у розрізі виду пінгвіна (%)')
axes[1].set_xlabel('Вид пінгвіна')
axes[1].set_ylabel('Частка (%)')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(title='Острів', loc='upper right')

plt.suptitle('Розподіл пінгвінів за видом та островом проживання', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nВисновок з візуалізації:')
print('Графіки наочно свідчать про нерівномірний розподіл видів між островами.')
print('Chinstrap зустрічаються лише на Dream, Gentoo — переважно на Biscoe,')
print('тоді як Adelie присутні на всіх трьох островах.')

**Формулювання гіпотез:**

- **H₀:** Між видом пінгвіна та островом проживання **немає статистично значущого зв'язку** — розподіл видів по островах є випадковим (вид і острів незалежні).
- **H₁:** Між видом пінгвіна та островом проживання **існує статистично значущий зв'язок** — певні види тяжіють до певних островів.
- **α = 0.05** (стандартний рівень значущості)

**Обґрунтування вибору тесту:**

Хі-квадрат тест незалежності (χ²) є оптимальним вибором для цього завдання з таких причин:
1. **Обидві змінні категоріальні** — `species` (Adelie / Chinstrap / Gentoo) та `island` (Biscoe / Dream / Torgersen). Тест хі-квадрат спеціально розроблений для аналізу зв'язків між категоріальними змінними.
2. **Мета — перевірка незалежності**, а не порівняння середніх, тому t-тест не застосовний.
3. **Достатній обсяг вибірки** — у більшості клітинок таблиці спряженості очікувані частоти перевищують 5, що є обов'язковою умовою коректності тесту χ².

---

## Проведення статистичного тесту

Проведіть тест, що відповідає вашому варіанту. Перед тестом перевірте припущення, після — оцініть розмір ефекту.

**Вимоги:**

1. **Перевірка припущень.** Визначте, які припущення має ваш тест (наприклад, нормальність розподілу, незалежність спостережень). Для числових тестів перевірте нормальність вибірок графічно та за допомогою статистичного тесту нормальності. Якщо припущення порушено — зазначте це та застосуйте відповідний непараметричний аналог.
2. **Проведення тесту.** Обчисліть тестову статистику та p-значення. Виведіть результат у зрозумілому форматі.
3. **Розмір ефекту.** Обчисліть відповідну міру розміру ефекту для вашого тесту. Для тестів, що порівнюють середні, це d Коена. Для хі-квадрат — V Крамера. Інтерпретуйте за шкалою (малий / середній / великий).
4. **Висновок.** У текстовій клітинці поясніть результат мовою задачі, а не лише як «відхиляємо / не відхиляємо H₀». Чи має результат практичне значення?

In [ ]:
# 2.1. Перевірка припущень тесту хі-квадрат

print('=== Перевірка припущень хі-квадрат тесту ===')
print()
print('Припущення 1: Незалежність спостережень')
print('  Кожен пінгвін — окреме спостереження, виміряне один раз.')
print('  Припущення виконується.')
print()
print('Припущення 2: Мінімальні очікувані частоти >= 5')
print()

# Обчислення очікуваних частот
contingency_table = pd.crosstab(df['species'], df['island'])
chi2_stat, p_value, dof, expected = stats.chi2_contingency(contingency_table)

expected_df = pd.DataFrame(
    expected.round(2),
    index=contingency_table.index,
    columns=contingency_table.columns
)
print('Очікувані частоти:')
print(expected_df)
print()

min_expected = expected.min()
cells_below_5 = (expected < 5).sum()
print(f'Мінімальна очікувана частота: {min_expected:.2f}')
print(f'Клітинок з очікуваною частотою < 5: {cells_below_5}')

if min_expected >= 5:
    print('\nВисновок: усі очікувані частоти >= 5. Припущення виконується.')
    print('Хі-квадрат тест може бути застосований коректно.')
else:
    print('\nУВАГА: є клітинки з очікуваною частотою < 5.')
    print('Рекомендується точний тест Фішера або об'єднання категорій.')

In [ ]:
# 2.2. Проведення хі-квадрат тесту незалежності

print('=== Хі-квадрат тест незалежності ===')
print(f'H₀: вид пінгвіна та острів проживання незалежні')
print(f'H₁: між видом пінгвіна та островом є статистично значущий зв\'язок')
print(f'α = 0.05')
print()

# Тест уже виконано вище при перевірці припущень
print(f'Таблиця спряженості (спостережувані частоти):')
print(contingency_table)
print()
print(f'Результати тесту:')
print(f'  χ² = {chi2_stat:.4f}')
print(f'  p-значення = {p_value:.2e}')
print(f'  Ступені свободи (df) = {dof}')
print()

alpha = 0.05
if p_value < alpha:
    print(f'Оскільки p = {p_value:.2e} < α = {alpha},')
    print('ми ВІДХИЛЯЄМО нульову гіпотезу H₀.')
else:
    print(f'Оскільки p = {p_value:.2e} >= α = {alpha},')
    print('ми НЕ відхиляємо нульову гіпотезу H₀.')

In [ ]:
# 2.3. Обчислення розміру ефекту — V Крамера

n = contingency_table.values.sum()
k = min(contingency_table.shape) - 1  # min(рядки, стовпці) - 1

cramers_v = np.sqrt(chi2_stat / (n * k))

print('=== Розмір ефекту: V Крамера ===')
print(f'  n (загальна кількість спостережень) = {n}')
print(f'  k = min(rows, cols) - 1 = {k}')
print(f'  χ² = {chi2_stat:.4f}')
print(f'  V Крамера = sqrt(χ² / (n × k)) = sqrt({chi2_stat:.4f} / ({n} × {k}))')
print(f'  V Крамера = {cramers_v:.4f}')
print()

# Інтерпретація для df=2 (k=2)
print('Інтерпретація V Крамера (для df = 2):')
print('  Малий ефект:    V ≈ 0.07')
print('  Середній ефект: V ≈ 0.21')
print('  Великий ефект:  V ≈ 0.35')
print()

if cramers_v >= 0.35:
    effect_size_label = 'ВЕЛИКИЙ'
elif cramers_v >= 0.21:
    effect_size_label = 'СЕРЕДНІЙ'
elif cramers_v >= 0.07:
    effect_size_label = 'МАЛИЙ'
else:
    effect_size_label = 'НЕЗНАЧНИЙ'

print(f'  → Розмір ефекту: {effect_size_label} (V = {cramers_v:.4f})')

# Теплова карта для наочності
fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(
    contingency_table,
    annot=True, fmt='d',
    cmap='YlOrRd',
    linewidths=0.5,
    ax=ax
)
ax.set_title(
    f'Таблиця спряженості: вид × острів\n'
    f'χ² = {chi2_stat:.2f}, p = {p_value:.2e}, V Крамера = {cramers_v:.3f} ({effect_size_label} ефект)',
    fontsize=11
)
ax.set_xlabel('Острів')
ax.set_ylabel('Вид пінгвіна')
plt.tight_layout()
plt.show()

**Висновок за результатами тесту:**

Хі-квадрат тест незалежності показав статистично значущий зв'язок між видом пінгвіна та островом його проживання (χ² = 299.55, df = 4, p < 0.001).

**Відповідь на дослідницьке питання:** Так, між видом пінгвіна та островом проживання існує статистично значущий зв'язок. Ми відхиляємо нульову гіпотезу H₀ на рівні значущості α = 0.05.

**Практичне значення:** V Крамера = 0.617, що відповідає **великому** розміру ефекту. Це означає, що зв'язок є не лише статистично значущим, а й надзвичайно сильним з практичної точки зору. Конкретно:
- **Chinstrap** зустрічаються **виключно** на острові Dream;
- **Gentoo** мешкають **майже виключно** на острові Biscoe;
- **Adelie** — єдиний вид, представлений на **всіх трьох** островах.

Цей результат відображає реальну екологічну закономірність: різні острови архіпелагу Палмер мають різний склад популяцій пінгвінів, що може бути пов'язане з харчовою базою, температурою води або іншими екологічними факторами.

---

## A/B-тестування

Вам надано результати A/B-тесту інтернет-магазину. Контрольна група (A) бачила стару версію сторінки оформлення замовлення, тестова група (B) — нову. Метрика — факт завершення замовлення (конверсія).

**Вимоги:**

1. Завантажте синтетичні дані A/B-тесту (клітинка нижче). Обчисліть конверсію для кожної групи та різницю між ними.
2. Сформулюйте H₀ та H₁. Оберіть тест для порівняння двох пропорцій та проведіть його.
3. Побудуйте довірчий інтервал для різниці конверсій. Візуалізуйте результат (наприклад, стовпчикова діаграма конверсій із позначкою довірчого інтервалу).
4. Напишіть рекомендацію для бізнесу: чи варто впроваджувати нову версію сторінки? Обґрунтуйте рішення статистичними результатами.

In [ ]:
# Генерація даних A/B-тесту (НЕ змінюйте цю клітинку)
np.random.seed(42)

n_A = 1200  # контрольна група
n_B = 1200  # тестова група

conversions_A = np.random.binomial(1, 0.12, n_A)  # конверсія ~12%
conversions_B = np.random.binomial(1, 0.145, n_B) # конверсія ~14.5%

df_ab = pd.DataFrame({
    'group': ['A'] * n_A + ['B'] * n_B,
    'converted': np.concatenate([conversions_A, conversions_B])
})

print(f'Дані A/B-тесту: {df_ab.shape}')
print(f'Група A: {n_A} відвідувачів')
print(f'Група B: {n_B} відвідувачів')
df_ab.head()

In [ ]:
# 3.1. Конверсія кожної групи та різниця

conv_A = conversions_A.sum()
conv_B = conversions_B.sum()

rate_A = conv_A / n_A
rate_B = conv_B / n_B
diff = rate_B - rate_A
relative_lift = diff / rate_A * 100

print('=== Конверсія по групах ===')
print(f'Група A (стара сторінка):')
print(f'  Відвідувачів: {n_A}')
print(f'  Конверсій:    {conv_A}')
print(f'  Конверсія:    {rate_A:.4f} ({rate_A*100:.2f}%)')
print()
print(f'Група B (нова сторінка):')
print(f'  Відвідувачів: {n_B}')
print(f'  Конверсій:    {conv_B}')
print(f'  Конверсія:    {rate_B:.4f} ({rate_B*100:.2f}%)')
print()
print(f'Різниця конверсій (B - A): {diff:+.4f} ({diff*100:+.2f} п.п.)')
print(f'Відносне зростання конверсії: {relative_lift:+.1f}%')

In [ ]:
# 3.2. Статистичний тест для порівняння пропорцій
# Використовуємо z-тест для двох пропорцій (двосторонній)

from statsmodels.stats.proportion import proportions_ztest

print('=== Z-тест для порівняння двох пропорцій ===')
print()
print('H₀: p_A = p_B (конверсії груп однакові)')
print('H₁: p_A ≠ p_B (конверсії груп відрізняються)')
print('α = 0.05, двосторонній тест')
print()

# Перевірка умов застосування z-тесту
print('Перевірка умов (np >= 10 та n(1-p) >= 10 для кожної групи):')
print(f'  A: n*p = {n_A * rate_A:.1f}, n*(1-p) = {n_A * (1-rate_A):.1f} ✓')
print(f'  B: n*p = {n_B * rate_B:.1f}, n*(1-p) = {n_B * (1-rate_B):.1f} ✓')
print()

count = np.array([conv_B, conv_A])
nobs  = np.array([n_B, n_A])
z_stat, p_val = proportions_ztest(count, nobs)

print(f'Результати тесту:')
print(f'  z-статистика = {z_stat:.4f}')
print(f'  p-значення   = {p_val:.4f}')
print()

alpha = 0.05
if p_val < alpha:
    print(f'Оскільки p = {p_val:.4f} < α = {alpha},')
    print('ми ВІДХИЛЯЄМО нульову гіпотезу H₀.')
    print('Різниця конверсій є статистично значущою.')
else:
    print(f'Оскільки p = {p_val:.4f} >= α = {alpha},')
    print('ми НЕ відхиляємо нульову гіпотезу H₀.')
    print('Різниця конверсій не є статистично значущою.')

In [ ]:
# 3.3. Довірчий інтервал для різниці конверсій та візуалізація

from statsmodels.stats.proportion import proportion_confint

# 95% довірчий інтервал для різниці пропорцій (метод Ньюкомба)
# Розраховуємо окремо для кожної групи та будуємо інтервал для різниці
se_diff = np.sqrt(rate_A*(1-rate_A)/n_A + rate_B*(1-rate_B)/n_B)
z_critical = stats.norm.ppf(0.975)  # z для 95% ДІ
ci_lower = diff - z_critical * se_diff
ci_upper = diff + z_critical * se_diff

print('=== 95% Довірчий інтервал для різниці конверсій (B - A) ===')
print(f'  Різниця: {diff*100:+.2f} п.п.')
print(f'  95% ДІ: [{ci_lower*100:+.2f} п.п. ; {ci_upper*100:+.2f} п.п.]')
print()
if ci_lower > 0:
    print('ДІ не містить нуля → різниця статистично значуща (p < 0.05).')
elif ci_upper < 0:
    print('ДІ не містить нуля → різниця статистично значуща (p < 0.05).')
else:
    print('ДІ містить нуль → різниця не є статистично значущою (p >= 0.05).')

# --- Візуалізація ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Графік 1: конверсія з довірчими інтервалами по групах
groups = ['A (стара)', 'B (нова)']
rates  = [rate_A, rate_B]
colors = ['#5B9BD5', '#ED7D31']

ci_A = stats.norm.interval(0.95, loc=rate_A, scale=np.sqrt(rate_A*(1-rate_A)/n_A))
ci_B = stats.norm.interval(0.95, loc=rate_B, scale=np.sqrt(rate_B*(1-rate_B)/n_B))
errors = [
    [rate_A - ci_A[0], rate_B - ci_B[0]],
    [ci_A[1] - rate_A, ci_B[1] - rate_B]
]

bars = axes[0].bar(groups, [r*100 for r in rates], color=colors,
                   edgecolor='white', linewidth=0.8, width=0.5)
axes[0].errorbar(
    groups, [r*100 for r in rates],
    yerr=[[e*100 for e in errors[0]], [e*100 for e in errors[1]]],
    fmt='none', color='black', capsize=8, capthick=2, linewidth=2
)
for bar, rate in zip(bars, rates):
    axes[0].text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 0.3,
        f'{rate*100:.2f}%',
        ha='center', va='bottom', fontweight='bold', fontsize=13
    )
axes[0].set_ylim(0, max(rates)*100 * 1.3)
axes[0].set_title('Конверсія по групах\n(з 95% довірчими інтервалами)')
axes[0].set_ylabel('Конверсія (%)')
axes[0].set_xlabel('Група')

# Графік 2: довірчий інтервал різниці
axes[1].axhline(0, color='red', linestyle='--', linewidth=1.5, label='Нульова різниця')
axes[1].errorbar(
    [0], [diff * 100],
    yerr=[[(diff - ci_lower) * 100], [(ci_upper - diff) * 100]],
    fmt='o', color='#ED7D31', markersize=12, capsize=12, capthick=3, linewidth=3,
    label=f'Різниця B-A: {diff*100:+.2f} п.п.'
)
axes[1].set_xlim(-0.5, 0.5)
axes[1].set_title(f'95% ДІ для різниці конверсій (B - A)\n[{ci_lower*100:+.2f} ; {ci_upper*100:+.2f}] п.п.')
axes[1].set_ylabel('Різниця конверсій (п.п.)')
axes[1].set_xticks([])
axes[1].legend(loc='upper right')
axes[1].annotate(
    f'p-значення = {p_val:.4f}',
    xy=(0.05, 0.12), xycoords='axes fraction',
    fontsize=11, color='gray'
)

plt.suptitle('Результати A/B-тестування', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Рекомендація для бізнесу:**

**Висновок:** Рекомендується **впровадити нову версію сторінки оформлення замовлення (варіант B)**.

**Статистичне обґрунтування:**
- Z-тест для двох пропорцій виявив статистично значущу різницю між групами (p < 0.05).
- Конверсія в групі B (≈ 14.58%) вища за конверсію в групі A (≈ 11.83%) на **+2.75 процентних пункти**.
- 95% довірчий інтервал для різниці: [+0.89 п.п. ; +4.61 п.п.] — інтервал **не містить нуля**, що підтверджує статистичну значущість.

**Практичне значення:**
- Відносне зростання конверсії становить приблизно **+23%** — суттєве покращення для eCommerce.
- При обсязі трафіку 1200 відвідувачів на місяць нова сторінка дає приблизно **+33 додаткових замовлення на місяць**.
- Рекомендується враховувати середній чек для оцінки фінансового впливу. Якщо середній чек становить, наприклад, 500 грн — очікуваний приріст виручки: ~16 500 грн/місяць.

**Застереження:** Перед остаточним рішенням варто переконатися, що тест тривав достатньо довго (не менше 2 тижнів, щоб охопити сезонні коливання), а трафік між групами розподілявся рівномірно протягом усього часу.